# DAC 2026 — V10 Structured CatBoost Risk Model

**Final frozen experiment notebook.** This notebook implements the preregistered A0–A3 V10 contract for the **Integrated Claim Risk and Audit Allocation System**.

Research boundary:

- A0: proven single-CatBoost representation control
- A1: A0 + seven predeclared conditional categorical interactions
- A2: A1 + fold-safe label-free support and peer-relative median/MAD features
- A3: A2 with `border_count=254`
- exact-signature grouped primary validation, independent grouped confirmation, then a same-seed GPU stochastic rerun
- no public-leaderboard-driven selection
- no `claim_id`, row index, source order, reconstructed labels, target encoding, external predictive data, stacker, boundary refiner, direct reranking, class weighting, or boundary weighting

If no candidate passes the full frozen gate, the notebook finishes normally with:

> **Modeling freeze. V3 remains champion.**

The notebook is intended for Kaggle **T4×2**. One T4 is used per fit deliberately; folds are trained sequentially so GPU topology and statistical treatment remain identical across arms.

In [1]:
# Environment bootstrap, imports, and hardware contract. No model training occurs in this cell.
import gc
import hashlib
import importlib.util
import json
import math
import os
import platform
import shutil
import subprocess
import sys
import time
import warnings
from pathlib import Path

if importlib.util.find_spec("catboost") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "catboost==1.2.8"])

import numpy as np
import pandas as pd
import sklearn
from catboost import CatBoostClassifier, __version__ as catboost_version
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.model_selection import StratifiedGroupKFold
from scipy.stats import spearmanr

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

IS_KAGGLE = Path("/kaggle/input").exists()
if not IS_KAGGLE:
    raise RuntimeError("V10 is frozen for Kaggle execution. Run this notebook in a Kaggle GPU session, not a local runtime.")

OUTPUT_DIR = Path("/kaggle/working/v10_artifacts")
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "fold_predictions").mkdir(exist_ok=True)

os.environ["PYTHONHASHSEED"] = "2026"


def _run_text(cmd):
    try:
        return subprocess.check_output(cmd, text=True, stderr=subprocess.STDOUT).strip()
    except Exception as exc:
        return f"UNAVAILABLE: {type(exc).__name__}: {exc}"

GPU_LINES = [x.strip() for x in _run_text([
    "nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"
]).splitlines() if x.strip() and not x.startswith("UNAVAILABLE")]
GPU_COUNT = len(GPU_LINES)
if GPU_COUNT < 1:
    raise RuntimeError("No NVIDIA GPU detected. V10 requires a Kaggle GPU accelerator; T4×2 is the target environment.")

# Frozen engineering choice: one GPU per CatBoost fit. Do not switch to multi-GPU training between arms.
ACTIVE_GPU_DEVICE = "0"

ENVIRONMENT = {
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "catboost": catboost_version,
    "gpu_count": GPU_COUNT,
    "gpus": GPU_LINES,
    "active_catboost_device": ACTIVE_GPU_DEVICE,
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
}
(OUTPUT_DIR / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2))
print(json.dumps(ENVIRONMENT, indent=2))
print("Artifact directory:", OUTPUT_DIR)

{
  "python": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.12.90+-x86_64-with-glibc2.35",
  "numpy": "2.0.2",
  "pandas": "2.3.3",
  "scikit_learn": "1.6.1",
  "catboost": "1.2.10",
  "gpu_count": 2,
  "gpus": [
    "Tesla T4, 15360 MiB, 580.159.04",
    "Tesla T4, 15360 MiB, 580.159.04"
  ],
  "active_catboost_device": "0",
  "cuda_visible_devices": null
}
Artifact directory: /kaggle/working/v10_artifacts


In [2]:
# Frozen V10 contract. Values in this cell are preregistered and must not be changed after model results are visible.
ID_COL = "claim_id"
TARGET_COL = "label"
CAPACITIES = (0.03, 0.05, 0.07)
PRIMARY_CAPACITY = 0.05
PRIMARY_PARTITION_SEED = 2026
SECONDARY_PARTITION_SEED = 9117
MODEL_BASE_SEED = 2127
BOOTSTRAP_SEED = 3407
BOOTSTRAP_REPS = 300
MIN_NONFRAUD_GROUP = 100
MIN_PEER_SUPPORT = 20
PEER_MAD_EPS = 1e-6
V3_ORIGINAL_TP5 = 7828
V3_NEAR_PARITY_TP5 = 7820

SEMANTIC_CATEGORICAL = {
    "kdkc", "dati2", "typeppk", "jkpst", "jnspelsep", "cmg", "severitylevel", "diagprimer"
}

HIGH_ORDER_INTERACTIONS = {
    "v10_x__dati2__typeppk__cmg": ("dati2", "typeppk", "cmg"),
    "v10_x__dati2__typeppk__cmg__diagprimer": ("dati2", "typeppk", "cmg", "diagprimer"),
    "v10_x__dati2__typeppk__cmg__severitylevel": ("dati2", "typeppk", "cmg", "severitylevel"),
    "v10_x__dati2__typeppk__jnspelsep__cmg": ("dati2", "typeppk", "jnspelsep", "cmg"),
    "v10_x__kdkc__typeppk__cmg__diagprimer": ("kdkc", "typeppk", "cmg", "diagprimer"),
    "v10_x__typeppk__cmg__diagprimer__severitylevel": ("typeppk", "cmg", "diagprimer", "severitylevel"),
    "v10_x__diagprimer__jnspelsep__severitylevel": ("diagprimer", "jnspelsep", "severitylevel"),
}

PEER_NUMERIC_VARS = (
    "log1p_los",
    "diagnosis_total",
    "procedure_total",
    "total_coded_burden",
    "burden_per_day",
    "total_nonzero_groups",
)

PEER_SPECS = (
    {
        "id": "dati2_typeppk_cmg",
        "primary": ("dati2", "typeppk", "cmg"),
        "fallbacks": (("typeppk", "cmg"),),
    },
    {
        "id": "dati2_typeppk_cmg_severity",
        "primary": ("dati2", "typeppk", "cmg", "severitylevel"),
        "fallbacks": (("dati2", "typeppk", "cmg"), ("typeppk", "cmg")),
    },
    {
        "id": "dati2_typeppk_service_cmg",
        "primary": ("dati2", "typeppk", "jnspelsep", "cmg"),
        "fallbacks": (("dati2", "typeppk", "cmg"), ("typeppk", "cmg")),
    },
    {
        "id": "typeppk_cmg_diag_severity",
        "primary": ("typeppk", "cmg", "diagprimer", "severitylevel"),
        "fallbacks": (("typeppk", "cmg", "diagprimer"), ("typeppk", "cmg")),
    },
    {
        "id": "diag_service_severity",
        "primary": ("diagprimer", "jnspelsep", "severitylevel"),
        "fallbacks": (("diagprimer", "jnspelsep"), ("diagprimer",)),
    },
)

CATBOOST_BASE_CONFIG = {
    "loss_function": "Logloss",
    "iterations": 2860,
    "learning_rate": 0.03,
    "depth": 8,
    "l2_leaf_reg": 8.0,
    "random_strength": 0.35,
    "bootstrap_type": "Bayesian",
    "bagging_temperature": 0.5,
    "max_ctr_complexity": 2,
    "task_type": "GPU",
    "allow_writing_files": False,
    "verbose": False,
}

ARM_SPECS = {
    "A0": {"description": "single CatBoost proven representation control", "level": 0, "border_count": 128},
    "A1": {"description": "A0 + seven frozen high-order categorical interactions", "level": 1, "border_count": 128},
    "A2": {"description": "A1 + fold-safe support and peer-relative representation", "level": 2, "border_count": 128},
    "A3": {"description": "A2 + border_count=254", "level": 3, "border_count": 254},
}

PRIMARY_GATES = {
    "min_delta_tp5": 20,
    "min_fold_wins": 4,
    "worst_fold_delta_tp": -5,
    "min_delta_ap": -0.001,
    "min_delta_nr3": -0.001,
    "min_delta_nr7": -0.001,
    "max_delta_brier": 0.002,
    "min_delta_protection5": -0.001,
    "min_delta_protection_edge": -0.002,
}
SECONDARY_GATES = {
    "min_delta_tp5": 10,
    "min_fold_wins": 3,
    "worst_fold_delta_tp": -5,
    "min_total_fold_wins": 7,
    "min_mean_tp_gain": 15,
}
COMPLEXITY_GATES = {"min_incremental_primary_tp": 8, "min_incremental_secondary_tp": 0}
RERUN_GATES = {
    "min_delta_tp5": 10,
    "min_fold_wins": 3,
    "worst_fold_delta_tp": -5,
    "min_repeat_overlap": 0.93,
    "max_relative_overlap_loss": 0.02,
}

FROZEN_CONTRACT = {
    "competition": "DAC 2026 Integrated Claim Risk and Audit Allocation System",
    "arms": ARM_SPECS,
    "primary_partition_seed": PRIMARY_PARTITION_SEED,
    "secondary_partition_seed": SECONDARY_PARTITION_SEED,
    "model_base_seed": MODEL_BASE_SEED,
    "capacities": CAPACITIES,
    "semantic_categorical": sorted(SEMANTIC_CATEGORICAL),
    "high_order_interactions": {k: list(v) for k, v in HIGH_ORDER_INTERACTIONS.items()},
    "peer_numeric_vars": list(PEER_NUMERIC_VARS),
    "peer_specs": list(PEER_SPECS),
    "min_peer_support": MIN_PEER_SUPPORT,
    "catboost_base_config": CATBOOST_BASE_CONFIG,
    "primary_gates": PRIMARY_GATES,
    "secondary_gates": SECONDARY_GATES,
    "complexity_gates": COMPLEXITY_GATES,
    "rerun_gates": RERUN_GATES,
    "submission_inference": "first primary run, mean probability across five outer-fold models",
    "stochastic_rerun_role": "confirmation only; never cherry-pick between stochastic runs",
    "forbidden_predictive_inputs": ["claim_id", "dataframe index", "source row position", "reconstructed labels", "external predictive data"],
    "stop_condition": "No V11. If V10 is not fully promoted, modeling freeze and V3 remains champion.",
}


def stable_json_hash(obj):
    payload = json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str).encode()
    return hashlib.sha256(payload).hexdigest()

FROZEN_CONTRACT_HASH = stable_json_hash(FROZEN_CONTRACT)
(OUTPUT_DIR / "v10_config.json").write_text(json.dumps({
    **FROZEN_CONTRACT,
    "frozen_contract_sha256": FROZEN_CONTRACT_HASH,
}, indent=2, default=str))
(OUTPUT_DIR / "a1_interaction_contract.json").write_text(json.dumps({
    "interactions": {k: list(v) for k, v in HIGH_ORDER_INTERACTIONS.items()},
    "construction": "row-wise string token only; no labels or population statistics",
}, indent=2))
(OUTPUT_DIR / "a2_peer_contract.json").write_text(json.dumps({
    "min_peer_support": MIN_PEER_SUPPORT,
    "peer_numeric_vars": list(PEER_NUMERIC_VARS),
    "peer_specs": list(PEER_SPECS),
    "statistics": ["median", "MAD", "residual", "robust_z", "abs_robust_z"],
    "fit_scope": "outer training fold only",
}, indent=2, default=str))
print("Frozen contract SHA-256:", FROZEN_CONTRACT_HASH)
print(pd.DataFrame([{"arm": k, **v} for k, v in ARM_SPECS.items()]))

Frozen contract SHA-256: d5987a85f287a24b515e93eadc7ff7895d459c67e772da93ac739a3ffbbc0051
  arm                                        description  level  border_count
0  A0      single CatBoost proven representation control      0           128
1  A1  A0 + seven frozen high-order categorical inter...      1           128
2  A2  A1 + fold-safe support and peer-relative repre...      2           128
3  A3                              A2 + border_count=254      3           254


## Integrity contract

The storage-order defect is treated only as a forensic warning. The notebook deliberately random-permutes each outer-training fold using the fixed model seed before fitting CatBoost, groups exact repeated signatures into one validation fold, and resolves exact top-K ties with a salted hash of `claim_id`. `claim_id` is therefore an evaluation key only, never a predictive feature.

All population-derived representation features are fitted on the outer training subset only. Validation and test rows never contribute support, frequency, median, MAD, or fallback statistics used to train a model.

In [3]:
# Official input discovery, byte hashing, schema validation, and row-order-safe alignment. No training occurs here.
FORBIDDEN_PATH_TOKENS = (
    "ground_truth", "labels_reconstructed", "reconstructed", "submission_50pct", "50pct_1_50pct_0"
)


def forbidden_path(path):
    text = str(path).lower()
    return any(token in text for token in FORBIDDEN_PATH_TOKENS)


def csv_schema(path):
    return pd.read_csv(path, nrows=0).columns.tolist()


def discover_competition_files():
    preferred = Path("/kaggle/input/competitions/dac-its-2026")
    parents = []
    if preferred.exists():
        parents.append(preferred)
    for train_path in Path("/kaggle/input").rglob("train.csv"):
        if forbidden_path(train_path):
            continue
        if train_path.parent not in parents:
            parents.append(train_path.parent)

    candidates = []
    for parent in parents:
        train_path = parent / "train.csv"
        test_path = parent / "test.csv"
        sample_path = parent / "sample_submission.csv"
        if not (train_path.exists() and test_path.exists() and sample_path.exists()):
            continue
        if any(forbidden_path(p) for p in (train_path, test_path, sample_path)):
            continue
        try:
            train_cols, test_cols, sample_cols = map(csv_schema, (train_path, test_path, sample_path))
        except Exception:
            continue
        if ID_COL not in train_cols or TARGET_COL not in train_cols:
            continue
        if set(test_cols) != set(train_cols) - {TARGET_COL}:
            continue
        if sample_cols != [ID_COL, "fraud_probability"]:
            continue
        parent_text = str(parent).lower()
        preference = 100 * int(parent == preferred) + 10 * int("dac" in parent_text) + 5 * int("2026" in parent_text)
        candidates.append((preference, train_path.stat().st_size, train_path, test_path, sample_path))

    if not candidates:
        raise FileNotFoundError("No schema-consistent official DAC train/test/sample trio found under /kaggle/input.")
    candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
    _, _, train_path, test_path, sample_path = candidates[0]
    return train_path, test_path, sample_path, candidates


def sha256_file(path, chunk_size=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def read_claim_csv(path):
    cols = csv_schema(path)
    string_cols = [c for c in ({ID_COL} | SEMANTIC_CATEGORICAL) if c in cols]
    return pd.read_csv(path, dtype={c: "string" for c in string_cols}, low_memory=False)

TRAIN_PATH, TEST_PATH, SAMPLE_PATH, INPUT_CANDIDATES = discover_competition_files()
train_raw = read_claim_csv(TRAIN_PATH)
test_raw_unaligned = read_claim_csv(TEST_PATH)
sample_submission = read_claim_csv(SAMPLE_PATH)

if list(sample_submission.columns) != [ID_COL, "fraud_probability"]:
    raise ValueError("Sample submission schema must be exactly claim_id,fraud_probability.")
if train_raw[ID_COL].duplicated().any() or test_raw_unaligned[ID_COL].duplicated().any():
    raise ValueError("claim_id must be unique within train and test.")
if set(train_raw[ID_COL].astype(str)) & set(test_raw_unaligned[ID_COL].astype(str)):
    raise ValueError("Train/test claim_id overlap detected; experiment contract invalid.")

# Align test once to official submission order. This is an ID join, never a predictive use of ID.
test_raw = (
    sample_submission[[ID_COL]].assign(_submission_order=np.arange(len(sample_submission)))
    .merge(test_raw_unaligned, on=ID_COL, how="left", validate="one_to_one", indicator=True)
    .sort_values("_submission_order").reset_index(drop=True)
)
if len(test_raw) != len(sample_submission) or not test_raw["_merge"].eq("both").all():
    raise ValueError("Test rows could not be aligned exactly to sample_submission claim IDs.")
test_raw = test_raw.drop(columns=["_submission_order", "_merge"])
del test_raw_unaligned

INPUT_HASHES = {
    "train.csv": sha256_file(TRAIN_PATH),
    "test.csv": sha256_file(TEST_PATH),
    "sample_submission.csv": sha256_file(SAMPLE_PATH),
}
(OUTPUT_DIR / "input_hashes.json").write_text(json.dumps({
    "selected_parent": str(TRAIN_PATH.parent),
    "files": INPUT_HASHES,
}, indent=2))

raw_y = pd.to_numeric(train_raw[TARGET_COL], errors="raise").astype(int).to_numpy()
label_transitions = int(np.count_nonzero(np.diff(raw_y)))
preflight_data = pd.DataFrame([
    {"check": "train_rows", "value": len(train_raw)},
    {"check": "test_rows", "value": len(test_raw)},
    {"check": "train_columns", "value": train_raw.shape[1]},
    {"check": "fraud_count", "value": int(raw_y.sum())},
    {"check": "fraud_prevalence", "value": float(raw_y.mean())},
    {"check": "raw_missing_cells_train", "value": int(train_raw.isna().sum().sum())},
    {"check": "raw_missing_cells_test", "value": int(test_raw.isna().sum().sum())},
    {"check": "kdkc_leading_zero_rows", "value": int(train_raw["kdkc"].astype("string").str.startswith("0").sum())},
    {"check": "dati2_leading_zero_rows", "value": int(train_raw["dati2"].astype("string").str.startswith("0").sum())},
    {"check": "stored_label_transitions_forensic_only", "value": label_transitions},
])
preflight_data.to_csv(OUTPUT_DIR / "preflight_data_audit.csv", index=False)
print("Selected official input:", TRAIN_PATH.parent)
print(preflight_data.to_string(index=False))
print("Input hashes:", json.dumps(INPUT_HASHES, indent=2))

Selected official input: /kaggle/input/competitions/dac-its-2026
                                 check         value
                            train_rows 160174.000000
                             test_rows  40043.000000
                         train_columns     53.000000
                           fraud_count  80203.000000
                      fraud_prevalence      0.500724
               raw_missing_cells_train      0.000000
                raw_missing_cells_test      0.000000
                kdkc_leading_zero_rows  46210.000000
               dati2_leading_zero_rows 160174.000000
stored_label_transitions_forensic_only      1.000000
Input hashes: {
  "train.csv": "e87c42e644e07745e1fdf270949a438e4d636e1497bbcc48b1bea032279ddc3b",
  "test.csv": "4df0cfa840ecf44bc2c6aa09270261f63e939341d276c7b75df1bde4f636b36e",
  "sample_submission.csv": "13d43ce8d76ef78c5cd66a60514db51a492fd58c048c0abee41969dda1e225b2"
}


In [4]:
# Metric contract, neutral tie handling, fairness/protection diagnostics, and sanity checks.
def make_tie_key(ids):
    out = np.empty(len(ids), dtype=np.uint64)
    for i, value in enumerate(pd.Series(ids, dtype="string").astype(str).tolist()):
        digest = hashlib.blake2b(("DAC2026_V10_TIE|" + value).encode(), digest_size=8).digest()
        out[i] = int.from_bytes(digest, byteorder="little", signed=False)
    return out


def topk_indices(scores, tie_key, capacity):
    scores = np.asarray(scores, dtype=float)
    tie_key = np.asarray(tie_key, dtype=np.uint64)
    if scores.ndim != 1 or len(scores) != len(tie_key):
        raise ValueError("scores and tie_key must be aligned one-dimensional arrays.")
    if not np.isfinite(scores).all():
        raise ValueError("Non-finite scores encountered.")
    k = int(math.floor(float(capacity) * len(scores)))
    if k <= 0:
        return np.array([], dtype=np.int64)
    return np.lexsort((tie_key, -scores))[:k]


def normalized_recall_components(y_true, scores, tie_key, capacity):
    y_true = np.asarray(y_true, dtype=np.int8)
    idx = topk_indices(scores, tie_key, capacity)
    k = len(idx)
    p = int(y_true.sum())
    tp = int(y_true[idx].sum()) if k else 0
    recall = tp / p if p else np.nan
    recall_max = min(1.0, k / p) if p else np.nan
    nr = recall / recall_max if p and recall_max else 0.0
    return {"K": k, "P": p, "TP": tp, "NR": float(nr), "precision": float(tp / k) if k else np.nan}


def age_groups(age):
    values = pd.to_numeric(pd.Series(age), errors="coerce").mask(lambda s: s < 0)
    return pd.cut(
        values,
        [-np.inf, 17, 39, 59, np.inf],
        labels=["0-17", "18-39", "40-59", "60+"],
        right=True,
    ).astype("string").fillna("Missing")


def protection_metrics(y_true, selected_idx, gender, age, min_nonfraud=MIN_NONFRAUD_GROUP):
    y_true = np.asarray(y_true, dtype=np.int8)
    selected = np.zeros(len(y_true), dtype=bool)
    selected[np.asarray(selected_idx, dtype=np.int64)] = True
    detail = []
    gaps = {}
    for attribute, values, levels in [
        ("gender", pd.Series(gender, dtype="string").str.strip().fillna("Missing"), None),
        ("age_group", age_groups(age), ["0-17", "18-39", "40-59", "60+"]),
    ]:
        if levels is None:
            levels = sorted(values.unique().tolist())
        fprs = []
        for level in levels:
            member = values.eq(level).to_numpy()
            legitimate = member & (y_true == 0)
            denominator = int(legitimate.sum())
            false_positive = int((legitimate & selected).sum())
            eligible = denominator >= min_nonfraud and str(level) != "Missing"
            fpr = false_positive / denominator if denominator else np.nan
            detail.append({
                "attribute": attribute,
                "group": str(level),
                "nonfraud": denominator,
                "legitimate_audits": false_positive,
                "FPR": fpr,
                "eligible": bool(eligible),
            })
            if eligible and np.isfinite(fpr):
                fprs.append(fpr)
        gaps[attribute] = float(max(fprs) - min(fprs)) if len(fprs) >= 2 else 0.0
    protection = 1.0 - max(gaps.values())
    return protection, gaps, pd.DataFrame(detail)


def evaluate_scores(y_true, scores, tie_key, gender, age, capacities=CAPACITIES):
    y_true = np.asarray(y_true, dtype=np.int8)
    scores = np.asarray(scores, dtype=float)
    if not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
        raise ValueError("Evaluation probabilities must be finite and in [0,1].")
    result = {
        "AveragePrecision": float(average_precision_score(y_true, scores)),
        "Brier": float(brier_score_loss(y_true, scores)),
    }
    fairness_tables = []
    for cap in capacities:
        comp = normalized_recall_components(y_true, scores, tie_key, cap)
        idx = topk_indices(scores, tie_key, cap)
        protection, gaps, detail = protection_metrics(y_true, idx, gender, age)
        suffix = f"{int(round(cap * 100))}"
        result.update({
            f"K{suffix}": comp["K"],
            f"TP{suffix}": comp["TP"],
            f"NR{suffix}": comp["NR"],
            f"Precision{suffix}": comp["precision"],
            f"LegitimateAudits{suffix}": comp["K"] - comp["TP"],
            f"Protection{suffix}": float(protection),
            f"GapGender{suffix}": float(gaps["gender"]),
            f"GapAge{suffix}": float(gaps["age_group"]),
        })
        detail.insert(0, "capacity", cap)
        fairness_tables.append(detail)
    return result, pd.concat(fairness_tables, ignore_index=True)


def fold_metric(y_true, scores, tie_key):
    comp = normalized_recall_components(y_true, scores, tie_key, PRIMARY_CAPACITY)
    return {
        "TP5": comp["TP"],
        "NR5": comp["NR"],
        "AP": float(average_precision_score(y_true, scores)),
        "Brier": float(brier_score_loss(y_true, scores)),
    }

# Metric sanity checks include permutation-invariant exact ties.
_demo_y = np.array([1, 0, 1, 0, 1, 0, 1, 0] * 10, dtype=np.int8)
_demo_ids = np.array([f"D{i:03d}" for i in range(len(_demo_y))])
_demo_key = make_tie_key(_demo_ids)
_demo_scores = np.ones(len(_demo_y)) * 0.5
_demo_sel = set(_demo_ids[topk_indices(_demo_scores, _demo_key, 0.05)].tolist())
_perm = np.random.default_rng(17).permutation(len(_demo_y))
_demo_sel_perm = set(_demo_ids[_perm][topk_indices(_demo_scores[_perm], _demo_key[_perm], 0.05)].tolist())
assert _demo_sel == _demo_sel_perm
assert normalized_recall_components(_demo_y, np.linspace(1, 0, len(_demo_y)), _demo_key, 0.05)["K"] == 4
print("Metric and label-neutral tie invariance checks: PASS")

Metric and label-neutral tie invariance checks: PASS


In [5]:
# Row-wise representation, invariant exact-signature groups, feature roster resolution, and grouped folds. No training occurs here.
def clean_claims(frame):
    out = frame.copy()
    for column in out.columns:
        if column == TARGET_COL:
            out[column] = pd.to_numeric(out[column], errors="raise")
        elif column in ({ID_COL} | SEMANTIC_CATEGORICAL) or str(out[column].dtype) in {"object", "string"}:
            values = out[column].astype("string").str.strip()
            out[column] = values.mask(values.eq(""), "__MISSING__").fillna("__MISSING__")
        else:
            out[column] = pd.to_numeric(out[column], errors="coerce")
    return out


def entropy_and_share(block):
    values = block.to_numpy(dtype=float)
    totals = values.sum(axis=1)
    shares = np.divide(values, totals[:, None], out=np.zeros_like(values), where=totals[:, None] > 0)
    entropy = -(np.where(shares > 0, shares * np.log(shares + 1e-12), 0.0)).sum(axis=1)
    max_share = np.divide(values.max(axis=1), totals, out=np.zeros(len(values)), where=totals > 0)
    return entropy, max_share


def engineer_static(frame):
    out = frame.copy()
    dx_cols = [c for c in out if c.startswith("dx2_")]
    proc_cols = [c for c in out if c.startswith("proc")]
    age = pd.to_numeric(out["umur"], errors="coerce")
    los = pd.to_numeric(out["los"], errors="coerce").clip(lower=0)

    out["age_band"] = age_groups(age)
    out["is_pediatric"] = age.le(17).fillna(False).astype(np.int8)
    out["is_older_adult"] = age.ge(60).fillna(False).astype(np.int8)
    out["zero_los"] = los.eq(0).fillna(False).astype(np.int8)
    out["log1p_los"] = np.log1p(los)
    out["age_x_log_los"] = age * out["log1p_los"]

    for prefix, cols in (("diagnosis", dx_cols), ("procedure", proc_cols)):
        block = out[cols].apply(pd.to_numeric, errors="coerce").fillna(0).clip(lower=0)
        out[f"{prefix}_total"] = block.sum(axis=1)
        out[f"{prefix}_nonzero_groups"] = block.ne(0).sum(axis=1)
        out[f"{prefix}_max_group"] = block.max(axis=1)
        ent, share = entropy_and_share(block)
        out[f"{prefix}_entropy"] = ent
        out[f"{prefix}_max_share"] = share

    out["total_coded_burden"] = out["diagnosis_total"] + out["procedure_total"]
    out["total_nonzero_groups"] = out["diagnosis_nonzero_groups"] + out["procedure_nonzero_groups"]
    out["log1p_coded_burden"] = np.log1p(out["total_coded_burden"])
    out["procedure_to_diagnosis"] = (out["procedure_total"] + 0.5) / (out["diagnosis_total"] + 0.5)
    out["burden_per_day"] = out["total_coded_burden"] / (los.fillna(0) + 1.0)
    out["procedure_per_day"] = out["procedure_total"] / (los.fillna(0) + 1.0)
    out["diagnosis_per_day"] = out["diagnosis_total"] / (los.fillna(0) + 1.0)

    missing_block = out.drop(columns=[c for c in (ID_COL, TARGET_COL) if c in out])
    missing_count = missing_block.isna().sum(axis=1)
    for column in missing_block.select_dtypes(include=["object", "string"]).columns:
        missing_count += missing_block[column].astype("string").eq("__MISSING__").astype(int)
    out["missing_field_count"] = missing_count

    for source in ("kdkc", "dati2"):
        out[f"{source}_prefix2"] = out[source].astype("string").str[:2]

    interactions = {
        "cmg__severity": ("cmg", "severitylevel"),
        "service__facility": ("jnspelsep", "typeppk"),
        "geo__facility": ("dati2", "typeppk"),
        "kdkc__dati2": ("kdkc", "dati2"),
        "diagnosis__cmg": ("diagprimer", "cmg"),
        "diagnosis__service": ("diagprimer", "jnspelsep"),
        "diagnosis__facility": ("diagprimer", "typeppk"),
        "cmg__facility": ("cmg", "typeppk"),
    }
    for name, cols in interactions.items():
        out[name] = out[cols[0]].astype("string").str.cat(out[cols[1]].astype("string"), sep="__")
    return out


def make_interaction_token(frame, columns):
    token = frame[columns[0]].astype("string")
    for column in columns[1:]:
        token = token.str.cat(frame[column].astype("string"), sep="\x1f")
    return token.fillna("__MISSING__")


def add_high_order_tokens(frame):
    out = frame.copy()
    for name, columns in HIGH_ORDER_INTERACTIONS.items():
        out[name] = make_interaction_token(out, columns)
    return out

train_clean = clean_claims(train_raw)
test_clean = clean_claims(test_raw)
train_static = engineer_static(train_clean)
test_static = engineer_static(test_clean)

y = pd.to_numeric(train_clean[TARGET_COL], errors="raise").astype(np.int8).to_numpy()
if not np.isin(y, [0, 1]).all():
    raise ValueError("Training label must be binary 0/1.")
TRAIN_TIE = make_tie_key(train_clean[ID_COL])
TEST_TIE = make_tie_key(test_clean[ID_COL])
GENDER = train_clean["jkpst"].astype("string").str.strip().to_numpy()
AGE = pd.to_numeric(train_clean["umur"], errors="coerce").to_numpy()

CONSTANT_COLUMNS = [
    c for c in train_static.columns
    if c not in {ID_COL, TARGET_COL} and train_static[c].nunique(dropna=False) <= 1
]
BASE_STATIC_FEATURES = [
    c for c in train_static.columns if c not in {ID_COL, TARGET_COL, *CONSTANT_COLUMNS}
]
BASE_CATEGORICAL = [
    c for c in BASE_STATIC_FEATURES
    if c in SEMANTIC_CATEGORICAL or str(train_static[c].dtype) in {"object", "string"}
]
FREQUENCY_BASE_COLUMNS = [
    c for c in ("kdkc", "dati2", "geo__facility", "kdkc__dati2", "diagnosis__facility", "diagnosis__cmg")
    if c in BASE_STATIC_FEATURES
]
FREQUENCY_FEATURES = [f"frequency__{c}" for c in FREQUENCY_BASE_COLUMNS]

# Add row-wise A1 tokens only after the A0 roster is resolved.
train_static = add_high_order_tokens(train_static)
test_static = add_high_order_tokens(test_static)
HIGH_ORDER_NAMES = list(HIGH_ORDER_INTERACTIONS)
SUPPORT_NUMERIC_FEATURES = []
SUPPORT_STATE_FEATURES = []
SUPPORT_FEATURES = []
for token in HIGH_ORDER_NAMES:
    numeric_names = [
        f"v10_support__{token}__count",
        f"v10_support__{token}__log1p",
        f"v10_support__{token}__frequency",
    ]
    state_name = f"v10_support__{token}__state"
    SUPPORT_NUMERIC_FEATURES.extend(numeric_names)
    SUPPORT_STATE_FEATURES.append(state_name)
    # Canonical support ordering mirrors add_support_blocks(): all four fields per interaction.
    SUPPORT_FEATURES.extend([*numeric_names, state_name])

PEER_FEATURES = []
for spec in PEER_SPECS:
    for var in PEER_NUMERIC_VARS:
        PEER_FEATURES.extend([
            f"v10_peer__{spec['id']}__{var}__median",
            f"v10_peer__{spec['id']}__{var}__mad",
            f"v10_peer__{spec['id']}__{var}__residual",
            f"v10_peer__{spec['id']}__{var}__robust_z",
            f"v10_peer__{spec['id']}__{var}__abs_robust_z",
        ])

A0_FEATURES = BASE_STATIC_FEATURES + FREQUENCY_FEATURES
A1_FEATURES = A0_FEATURES + HIGH_ORDER_NAMES
A2_FEATURES = A1_FEATURES + SUPPORT_FEATURES + PEER_FEATURES
FEATURES_BY_ARM = {"A0": A0_FEATURES, "A1": A1_FEATURES, "A2": A2_FEATURES, "A3": A2_FEATURES}
CATEGORICAL_BY_ARM = {
    "A0": BASE_CATEGORICAL,
    "A1": BASE_CATEGORICAL + HIGH_ORDER_NAMES,
    "A2": BASE_CATEGORICAL + HIGH_ORDER_NAMES + SUPPORT_STATE_FEATURES,
    "A3": BASE_CATEGORICAL + HIGH_ORDER_NAMES + SUPPORT_STATE_FEATURES,
}

for arm, features in FEATURES_BY_ARM.items():
    if ID_COL in features or TARGET_COL in features:
        raise AssertionError(f"Forbidden identifier/target in {arm} feature list.")
    if any("row_position" in c.lower() or c.lower() == "index" for c in features):
        raise AssertionError(f"Forbidden row/index feature in {arm}.")
    if len(features) != len(set(features)):
        raise AssertionError(f"Duplicate feature name in {arm}.")
    categorical = CATEGORICAL_BY_ARM[arm]
    if len(categorical) != len(set(categorical)):
        raise AssertionError(f"Duplicate categorical feature name in {arm}.")
    missing_categorical = [c for c in categorical if c not in features]
    if missing_categorical:
        raise AssertionError(f"Categorical features missing from {arm} roster: {missing_categorical}")

# Exact-signature grouping is invariant across A0-A3 and excludes ID/label.
signature_frame = train_clean.drop(columns=[ID_COL, TARGET_COL], errors="ignore")
SIGNATURE_GROUPS = pd.util.hash_pandas_object(signature_frame, index=False).to_numpy(dtype=np.uint64)
sig_counts = pd.Series(SIGNATURE_GROUPS).value_counts()
repeated_groups = int((sig_counts > 1).sum())
rows_in_repeated = int(sig_counts[sig_counts > 1].sum())
if repeated_groups != 6747 or rows_in_repeated != 17906:
    raise RuntimeError(
        f"Dataset signature contract mismatch: repeated_groups={repeated_groups}, rows_in_repeated={rows_in_repeated}; expected 6747/17906."
    )


def make_grouped_partition(seed, name):
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    fold_id = np.full(len(y), -1, dtype=np.int8)
    splits = []
    rows = []
    for fold, (fit_idx, valid_idx) in enumerate(splitter.split(train_static, y, groups=SIGNATURE_GROUPS)):
        fold_id[valid_idx] = fold
        overlap = len(set(SIGNATURE_GROUPS[fit_idx]).intersection(set(SIGNATURE_GROUPS[valid_idx])))
        if overlap:
            raise RuntimeError(f"{name} fold {fold}: exact-signature leakage detected.")
        splits.append((fit_idx, valid_idx))
        rows.append({
            "partition": name,
            "fold": fold,
            "train_n": len(fit_idx),
            "valid_n": len(valid_idx),
            "valid_fraud_rate": float(y[valid_idx].mean()),
            "signature_overlap": overlap,
        })
    if (fold_id < 0).any():
        raise RuntimeError(f"{name} partition did not assign every row.")
    return splits, fold_id, pd.DataFrame(rows)

PRIMARY_SPLITS, PRIMARY_FOLD_ID, PRIMARY_FOLD_SUMMARY = make_grouped_partition(PRIMARY_PARTITION_SEED, "primary")
SECONDARY_SPLITS, SECONDARY_FOLD_ID, SECONDARY_FOLD_SUMMARY = make_grouped_partition(SECONDARY_PARTITION_SEED, "secondary")
if np.array_equal(PRIMARY_FOLD_ID, SECONDARY_FOLD_ID):
    raise RuntimeError("Primary and secondary grouped partitions are unexpectedly identical.")

pd.DataFrame({ID_COL: train_clean[ID_COL].astype(str), "fold": PRIMARY_FOLD_ID}).to_csv(OUTPUT_DIR / "fold_assignments_primary.csv", index=False)
pd.DataFrame({ID_COL: train_clean[ID_COL].astype(str), "fold": SECONDARY_FOLD_ID}).to_csv(OUTPUT_DIR / "fold_assignments_secondary.csv", index=False)
PRIMARY_FOLD_SUMMARY.to_csv(OUTPUT_DIR / "fold_summary_primary.csv", index=False)
SECONDARY_FOLD_SUMMARY.to_csv(OUTPUT_DIR / "fold_summary_secondary.csv", index=False)

FEATURE_MANIFEST = {
    arm: {
        "features": FEATURES_BY_ARM[arm],
        "categorical": CATEGORICAL_BY_ARM[arm],
        "feature_count": len(FEATURES_BY_ARM[arm]),
        "border_count": ARM_SPECS[arm]["border_count"],
    }
    for arm in ARM_SPECS
}
(OUTPUT_DIR / "feature_manifest.json").write_text(json.dumps(FEATURE_MANIFEST, indent=2))
for arm in ARM_SPECS:
    (OUTPUT_DIR / f"{arm.lower()}_feature_list.txt").write_text("\n".join(FEATURES_BY_ARM[arm]) + "\n")

RESOLVED_EXPERIMENT = {
    "frozen_contract_sha256": FROZEN_CONTRACT_HASH,
    "input_hashes": INPUT_HASHES,
    "constant_columns_excluded": CONSTANT_COLUMNS,
    "frequency_base_columns": FREQUENCY_BASE_COLUMNS,
    "feature_manifest": FEATURE_MANIFEST,
    "repeated_exact_signature_groups": repeated_groups,
    "rows_in_repeated_exact_signatures": rows_in_repeated,
    "primary_fold_counts": PRIMARY_FOLD_SUMMARY.to_dict(orient="records"),
    "secondary_fold_counts": SECONDARY_FOLD_SUMMARY.to_dict(orient="records"),
}
RESOLVED_EXPERIMENT_HASH = stable_json_hash(RESOLVED_EXPERIMENT)
RESOLVED_EXPERIMENT["resolved_experiment_sha256"] = RESOLVED_EXPERIMENT_HASH
(OUTPUT_DIR / "resolved_experiment.json").write_text(json.dumps(RESOLVED_EXPERIMENT, indent=2, default=str))

print("Resolved experiment SHA-256:", RESOLVED_EXPERIMENT_HASH)
print("Feature counts:", {arm: len(features) for arm, features in FEATURES_BY_ARM.items()})
print(PRIMARY_FOLD_SUMMARY.to_string(index=False))
print(SECONDARY_FOLD_SUMMARY.to_string(index=False))

Resolved experiment SHA-256: 01edf63d5cfef78400b66765ca0e34c704b20811f57e47f8a11421f031a8b9b0
Feature counts: {'A0': 87, 'A1': 94, 'A2': 272, 'A3': 272}
partition  fold  train_n  valid_n  valid_fraud_rate  signature_overlap
  primary     0   128063    32111          0.500607                  0
  primary     1   128058    32116          0.504639                  0
  primary     2   128158    32016          0.496408                  0
  primary     3   128361    31813          0.496967                  0
  primary     4   128056    32118          0.504950                  0
partition  fold  train_n  valid_n  valid_fraud_rate  signature_overlap
secondary     0   128021    32153          0.501073                  0
secondary     1   128139    32035          0.499298                  0
secondary     2   128104    32070          0.501933                  0
secondary     3   128280    31894          0.498997                  0
secondary     4   128152    32022          0.502311               

In [6]:
# Fold-safe population features: proven frequency block, A2 support, peer median/MAD, and fallback tracking.
def make_key(frame, columns):
    key = frame[columns[0]].astype("string")
    for column in columns[1:]:
        key = key.str.cat(frame[column].astype("string"), sep="\x1f")
    return key.fillna("__MISSING__")


def add_existing_frequency_blocks(fit_frame, valid_frame, test_frame):
    fit_out = pd.DataFrame(index=fit_frame.index)
    valid_out = pd.DataFrame(index=valid_frame.index)
    test_out = pd.DataFrame(index=test_frame.index)
    for column in FREQUENCY_BASE_COLUMNS:
        mapping = fit_frame[column].astype("string").value_counts(dropna=False) / max(len(fit_frame), 1)
        name = f"frequency__{column}"
        fit_out[name] = fit_frame[column].astype("string").map(mapping).fillna(0).astype(np.float32)
        valid_out[name] = valid_frame[column].astype("string").map(mapping).fillna(0).astype(np.float32)
        test_out[name] = test_frame[column].astype("string").map(mapping).fillna(0).astype(np.float32)
    return fit_out, valid_out, test_out


def add_support_blocks(fit_frame, valid_frame, test_frame, partition, fold):
    outputs = {"fit": pd.DataFrame(index=fit_frame.index), "valid": pd.DataFrame(index=valid_frame.index), "test": pd.DataFrame(index=test_frame.index)}
    frames = {"fit": fit_frame, "valid": valid_frame, "test": test_frame}
    profiles = []
    for token in HIGH_ORDER_NAMES:
        counts = fit_frame[token].astype("string").value_counts(dropna=False)
        profiles.append({
            "partition": partition,
            "fold": fold,
            "interaction": token,
            "n_levels_train": int(len(counts)),
            "median_support": float(counts.median()) if len(counts) else 0.0,
            "max_support": int(counts.max()) if len(counts) else 0,
            "levels_support_lt20": int((counts < MIN_PEER_SUPPORT).sum()),
            "rows_in_support_lt20": int(counts[counts < MIN_PEER_SUPPORT].sum()),
        })
        for split_name, frame in frames.items():
            support = frame[token].astype("string").map(counts).fillna(0).astype(np.float32)
            outputs[split_name][f"v10_support__{token}__count"] = support
            outputs[split_name][f"v10_support__{token}__log1p"] = np.log1p(support).astype(np.float32)
            outputs[split_name][f"v10_support__{token}__frequency"] = (support / max(len(fit_frame), 1)).astype(np.float32)
            state = np.where(support.to_numpy() == 0, "UNSEEN", np.where(support.to_numpy() < MIN_PEER_SUPPORT, "LOW", "ADEQUATE"))
            outputs[split_name][f"v10_support__{token}__state"] = pd.Series(state, index=frame.index, dtype="string")
    for split_name, block in outputs.items():
        if list(block.columns) != SUPPORT_FEATURES:
            raise RuntimeError(
                f"Support feature roster mismatch for {partition} fold {fold} {split_name}: "
                "constructor order diverged from the frozen resolved feature contract."
            )
    return outputs["fit"], outputs["valid"], outputs["test"], profiles


def group_stat_maps(fit_frame, columns, variable, cache):
    cache_key = (tuple(columns), variable)
    if cache_key in cache:
        return cache[cache_key]
    key = make_key(fit_frame, columns)
    value = pd.to_numeric(fit_frame[variable], errors="coerce").astype(float)
    tmp = pd.DataFrame({"key": key.to_numpy(), "value": value.to_numpy()})
    grouped = tmp.groupby("key", sort=False, observed=True)["value"]
    count = grouped.size()
    median = grouped.median()
    mapped_median = key.map(median).to_numpy(dtype=float)
    abs_dev = np.abs(value.to_numpy(dtype=float) - mapped_median)
    mad = pd.DataFrame({"key": key.to_numpy(), "dev": abs_dev}).groupby("key", sort=False, observed=True)["dev"].median()
    cache[cache_key] = (count, median, mad)
    return cache[cache_key]


def peer_block_for_frame(fit_frame, apply_frame, spec, variable, stats_cache, key_cache, apply_name):
    n = len(apply_frame)
    resolved_median = np.full(n, np.nan, dtype=np.float64)
    resolved_mad = np.full(n, np.nan, dtype=np.float64)
    resolved_level = np.full(n, -1, dtype=np.int16)
    chains = (spec["primary"],) + tuple(spec["fallbacks"])

    for level, columns in enumerate(chains):
        count_map, median_map, mad_map = group_stat_maps(fit_frame, columns, variable, stats_cache)
        key_id = (apply_name, tuple(columns))
        if key_id not in key_cache:
            key_cache[key_id] = make_key(apply_frame, columns)
        key = key_cache[key_id]
        count = key.map(count_map).fillna(0).to_numpy(dtype=float)
        median = key.map(median_map).to_numpy(dtype=float)
        mad = key.map(mad_map).to_numpy(dtype=float)
        unresolved = resolved_level < 0
        usable = unresolved & (count >= MIN_PEER_SUPPORT) & np.isfinite(median) & np.isfinite(mad) & (mad > PEER_MAD_EPS)
        resolved_median[usable] = median[usable]
        resolved_mad[usable] = mad[usable]
        resolved_level[usable] = level

    fit_value = pd.to_numeric(fit_frame[variable], errors="coerce").to_numpy(dtype=float)
    global_median = float(np.nanmedian(fit_value))
    global_mad = float(np.nanmedian(np.abs(fit_value - global_median)))
    if not np.isfinite(global_median):
        raise RuntimeError(f"Global peer median is non-finite for {variable}.")
    if not np.isfinite(global_mad):
        global_mad = 0.0
    unresolved = resolved_level < 0
    resolved_median[unresolved] = global_median
    resolved_mad[unresolved] = global_mad
    resolved_level[unresolved] = len(chains)

    value = pd.to_numeric(apply_frame[variable], errors="coerce").to_numpy(dtype=float)
    value = np.where(np.isfinite(value), value, global_median)
    residual = value - resolved_median
    denom = np.maximum(1.4826 * resolved_mad, PEER_MAD_EPS)
    robust_z = residual / denom
    if not np.isfinite(robust_z).all():
        raise RuntimeError(f"Non-finite peer robust_z for {spec['id']} / {variable} / {apply_name}.")

    prefix = f"v10_peer__{spec['id']}__{variable}"
    block = pd.DataFrame({
        f"{prefix}__median": resolved_median.astype(np.float32),
        f"{prefix}__mad": resolved_mad.astype(np.float32),
        f"{prefix}__residual": residual.astype(np.float32),
        f"{prefix}__robust_z": robust_z.astype(np.float32),
        f"{prefix}__abs_robust_z": np.abs(robust_z).astype(np.float32),
    }, index=apply_frame.index)
    counts = np.bincount(resolved_level, minlength=len(chains) + 1)
    usage = []
    for level, count in enumerate(counts):
        label = "global" if level == len(chains) else ("primary" if level == 0 else f"fallback_{level}")
        usage.append({
            "peer": spec["id"], "variable": variable, "dataset": apply_name,
            "resolution_level": label, "rows": int(count), "share": float(count / max(n, 1)),
        })
    return block, usage


def add_peer_blocks(fit_frame, valid_frame, test_frame, partition, fold):
    outputs = {"fit": pd.DataFrame(index=fit_frame.index), "valid": pd.DataFrame(index=valid_frame.index), "test": pd.DataFrame(index=test_frame.index)}
    stats_cache = {}
    key_cache = {}
    usage_rows = []
    for spec in PEER_SPECS:
        for variable in PEER_NUMERIC_VARS:
            for split_name, frame in (("fit", fit_frame), ("valid", valid_frame), ("test", test_frame)):
                block, usage = peer_block_for_frame(
                    fit_frame, frame, spec, variable, stats_cache, key_cache, split_name
                )
                outputs[split_name] = pd.concat([outputs[split_name], block], axis=1)
                for row in usage:
                    row.update({"partition": partition, "fold": fold})
                usage_rows.extend(usage)
    for split_name, block in outputs.items():
        if list(block.columns) != PEER_FEATURES:
            raise RuntimeError(f"Peer feature roster mismatch for {partition} fold {fold} {split_name}.")
    return outputs["fit"], outputs["valid"], outputs["test"], usage_rows


def build_fold_views(fit_idx, valid_idx, partition, fold, requested_arms):
    max_level = max(ARM_SPECS[a]["level"] for a in requested_arms)
    fit_static = train_static.iloc[fit_idx]
    valid_static = train_static.iloc[valid_idx]
    test_fold_static = test_static

    freq_fit, freq_valid, freq_test = add_existing_frequency_blocks(fit_static, valid_static, test_fold_static)
    a0_fit = pd.concat([fit_static[BASE_STATIC_FEATURES], freq_fit], axis=1)
    a0_valid = pd.concat([valid_static[BASE_STATIC_FEATURES], freq_valid], axis=1)
    a0_test = pd.concat([test_fold_static[BASE_STATIC_FEATURES], freq_test], axis=1)
    views = {"A0": (a0_fit, a0_valid, a0_test)}
    support_profiles, fallback_usage = [], []

    if max_level >= 1:
        a1_fit = pd.concat([a0_fit, fit_static[HIGH_ORDER_NAMES]], axis=1)
        a1_valid = pd.concat([a0_valid, valid_static[HIGH_ORDER_NAMES]], axis=1)
        a1_test = pd.concat([a0_test, test_fold_static[HIGH_ORDER_NAMES]], axis=1)
        views["A1"] = (a1_fit, a1_valid, a1_test)

    if max_level >= 2:
        sup_fit, sup_valid, sup_test, profiles = add_support_blocks(
            fit_static, valid_static, test_fold_static, partition, fold
        )
        peer_fit, peer_valid, peer_test, usage = add_peer_blocks(
            fit_static, valid_static, test_fold_static, partition, fold
        )
        support_profiles.extend(profiles)
        fallback_usage.extend(usage)
        a2_fit = pd.concat([a1_fit, sup_fit, peer_fit], axis=1)
        a2_valid = pd.concat([a1_valid, sup_valid, peer_valid], axis=1)
        a2_test = pd.concat([a1_test, sup_test, peer_test], axis=1)
        views["A2"] = (a2_fit, a2_valid, a2_test)
        views["A3"] = views["A2"]

    for arm in requested_arms:
        expected = FEATURES_BY_ARM[arm]
        frame_triplet = views["A2"] if arm == "A3" else views[arm]
        for split_name, frame in zip(("fit", "valid", "test"), frame_triplet):
            actual = list(frame.columns)
            if actual != expected:
                missing = [c for c in expected if c not in actual]
                extra = [c for c in actual if c not in expected]
                first_mismatch = next(
                    (i for i, (exp, got) in enumerate(zip(expected, actual)) if exp != got),
                    None,
                )
                raise RuntimeError(
                    f"Feature contract mismatch for {partition} fold {fold} {arm} {split_name}: "
                    f"expected_n={len(expected)}, actual_n={len(actual)}, "
                    f"missing={missing[:5]}, extra={extra[:5]}, first_mismatch={first_mismatch}."
                )
    return views, support_profiles, fallback_usage

print("Fold-safe feature constructors defined. A2/A3 feature count:", len(A2_FEATURES))

Fold-safe feature constructors defined. A2/A3 feature count: 272


In [7]:
# Training, partition execution, and artifact persistence. Each fold is seed-permuted before CatBoost sees it.
def assert_contract_unchanged():
    current = stable_json_hash(FROZEN_CONTRACT)
    if current != FROZEN_CONTRACT_HASH:
        raise RuntimeError("Frozen V10 contract mutated after preflight; aborting experiment.")


def catboost_ready(frame, categorical):
    out = frame.copy()
    categorical_set = set(categorical)
    for column in out.columns:
        if column in categorical_set:
            out[column] = out[column].astype("string").fillna("__MISSING__").astype(str)
        else:
            out[column] = pd.to_numeric(out[column], errors="coerce").astype(np.float32)
    return out


def fit_catboost_predict(fit_frame, fit_y, valid_frame, test_frame, arm, fold):
    assert_contract_unchanged()
    seed = MODEL_BASE_SEED + int(fold)
    features = FEATURES_BY_ARM[arm]
    categorical = CATEGORICAL_BY_ARM[arm]
    border_count = ARM_SPECS[arm]["border_count"]

    # Random permutation neutralizes stored source order and is identical across arms for a given fold.
    order = np.random.default_rng(seed).permutation(len(fit_y))
    fit_ready = catboost_ready(fit_frame.iloc[order][features].reset_index(drop=True), categorical)
    y_ready = np.asarray(fit_y, dtype=np.int8)[order]
    valid_ready = catboost_ready(valid_frame[features], categorical)
    test_ready = catboost_ready(test_frame[features], categorical)

    params = dict(CATBOOST_BASE_CONFIG)
    params.update({
        "random_seed": seed,
        "border_count": border_count,
        "devices": ACTIVE_GPU_DEVICE,
    })
    model = CatBoostClassifier(**params)
    started = time.perf_counter()
    try:
        model.fit(fit_ready, y_ready, cat_features=categorical, verbose=False)
        valid_pred = model.predict_proba(valid_ready)[:, 1].astype(np.float64)
        test_pred = model.predict_proba(test_ready)[:, 1].astype(np.float64)
    except Exception as exc:
        raise RuntimeError(
            f"CatBoost failed for arm={arm}, fold={fold}, device={ACTIVE_GPU_DEVICE}. "
            "The frozen model configuration is not auto-modified on failure; repair only the Kaggle runtime/environment. "
            f"Original error: {type(exc).__name__}: {exc}"
        ) from exc
    elapsed = time.perf_counter() - started
    if not np.isfinite(valid_pred).all() or not np.isfinite(test_pred).all():
        raise RuntimeError(f"Non-finite predictions from {arm} fold {fold}.")
    if ((valid_pred < 0) | (valid_pred > 1)).any() or ((test_pred < 0) | (test_pred > 1)).any():
        raise RuntimeError(f"Out-of-range probabilities from {arm} fold {fold}.")
    del model, fit_ready, valid_ready, test_ready, y_ready
    gc.collect()
    return valid_pred, test_pred, elapsed


def run_partition(partition_name, splits, requested_arms, run_label="run1"):
    requested_arms = list(requested_arms)
    if "A0" not in requested_arms:
        requested_arms = ["A0"] + requested_arms
    requested_arms = sorted(set(requested_arms), key=lambda a: ARM_SPECS[a]["level"])
    print(f"\n=== {partition_name.upper()} / {run_label} / arms={requested_arms} ===")
    scores = {
        arm: {"oof": np.full(len(y), np.nan, dtype=np.float64), "test_folds": [], "train_seconds": 0.0}
        for arm in requested_arms
    }
    fold_rows, support_profiles, fallback_usage = [], [], []

    for fold, (fit_idx, valid_idx) in enumerate(splits):
        print(f"[{partition_name}/{run_label}] fold {fold + 1}/5 | build fold-safe representation")
        views, profiles, usage = build_fold_views(fit_idx, valid_idx, partition_name, fold, requested_arms)
        support_profiles.extend(profiles)
        fallback_usage.extend(usage)
        for arm in requested_arms:
            view_key = "A2" if arm == "A3" else arm
            fit_frame, valid_frame, test_frame = views[view_key]
            valid_pred, test_pred, elapsed = fit_catboost_predict(
                fit_frame, y[fit_idx], valid_frame, test_frame, arm, fold
            )
            scores[arm]["oof"][valid_idx] = valid_pred
            scores[arm]["test_folds"].append(test_pred.astype(np.float32))
            scores[arm]["train_seconds"] += elapsed
            fm = fold_metric(y[valid_idx], valid_pred, TRAIN_TIE[valid_idx])
            fold_rows.append({
                "partition": partition_name,
                "run": run_label,
                "arm": arm,
                "fold": fold,
                "seed": MODEL_BASE_SEED + fold,
                "feature_count": len(FEATURES_BY_ARM[arm]),
                "border_count": ARM_SPECS[arm]["border_count"],
                "train_seconds": elapsed,
                **fm,
            })
            np.savez_compressed(
                OUTPUT_DIR / "fold_predictions" / f"{partition_name}_{run_label}_{arm}_fold{fold}.npz",
                valid_idx=valid_idx.astype(np.int32), valid_pred=valid_pred.astype(np.float32), test_pred=test_pred.astype(np.float32)
            )
            print(
                f"  {arm}: {elapsed:7.1f}s | TP5={fm['TP5']:4d} | NR5={fm['NR5']:.6f} | AP={fm['AP']:.6f}"
            )
        del views
        gc.collect()

    for arm in requested_arms:
        if np.isnan(scores[arm]["oof"]).any():
            raise RuntimeError(f"{partition_name}/{run_label}/{arm}: incomplete OOF predictions.")
        test_folds = np.vstack(scores[arm]["test_folds"]).astype(np.float32)
        scores[arm]["test_folds"] = test_folds
        scores[arm]["test"] = test_folds.mean(axis=0, dtype=np.float64)
        np.savez_compressed(
            OUTPUT_DIR / f"predictions_{partition_name}_{run_label}_{arm}.npz",
            oof=scores[arm]["oof"].astype(np.float32),
            test=scores[arm]["test"].astype(np.float32),
            test_folds=test_folds,
        )

    fold_df = pd.DataFrame(fold_rows)
    if support_profiles:
        pd.DataFrame(support_profiles).to_csv(OUTPUT_DIR / f"peer_support_profile_{partition_name}_{run_label}.csv", index=False)
    if fallback_usage:
        pd.DataFrame(fallback_usage).to_csv(OUTPUT_DIR / f"fallback_usage_{partition_name}_{run_label}.csv", index=False)
    fold_df.to_csv(OUTPUT_DIR / f"fold_metrics_{partition_name}_{run_label}.csv", index=False)
    return scores, fold_df

print("Training engine defined. Canonical CatBoost device:", ACTIVE_GPU_DEVICE)

Training engine defined. Canonical CatBoost device: 0


In [8]:
# Evaluation, portfolio/cutoff diagnostics, conditional exact-signature-cluster bootstrap, and gate helpers.
def evaluate_partition_scores(partition_name, run_label, scores, fold_df):
    registry_rows, fairness_frames = [], []
    for arm, bundle in scores.items():
        metrics, fairness = evaluate_scores(y, bundle["oof"], TRAIN_TIE, GENDER, AGE)
        fairness.insert(0, "arm", arm)
        fairness.insert(0, "run", run_label)
        fairness.insert(0, "partition", partition_name)
        fairness_frames.append(fairness)
        registry_rows.append({
            "partition": partition_name,
            "run": run_label,
            "arm": arm,
            "feature_count": len(FEATURES_BY_ARM[arm]),
            "border_count": ARM_SPECS[arm]["border_count"],
            "train_seconds": bundle["train_seconds"],
            **metrics,
        })
    registry = pd.DataFrame(registry_rows).sort_values("arm").reset_index(drop=True)
    fairness_df = pd.concat(fairness_frames, ignore_index=True)
    registry.to_csv(OUTPUT_DIR / f"candidate_registry_{partition_name}_{run_label}.csv", index=False)
    fairness_df.to_csv(OUTPUT_DIR / f"fairness_{partition_name}_{run_label}.csv", index=False)
    return registry, fairness_df


def rank_positions(scores, tie_key):
    order = np.lexsort((np.asarray(tie_key, dtype=np.uint64), -np.asarray(scores, dtype=float)))
    rank = np.empty(len(order), dtype=np.int32)
    rank[order] = np.arange(len(order), dtype=np.int32)
    return rank, order


def portfolio_diagnostic(base_scores, candidate_scores, y_true, tie_key, ids, arm, source="oof", capacity=0.05):
    base_idx = topk_indices(base_scores, tie_key, capacity)
    cand_idx = topk_indices(candidate_scores, tie_key, capacity)
    base_set, cand_set = set(base_idx.tolist()), set(cand_idx.tolist())
    out_only = sorted(base_set - cand_set)
    in_only = sorted(cand_set - base_set)
    k = len(base_idx)
    base_rank, base_order = rank_positions(base_scores, tie_key)
    cand_rank, cand_order = rank_positions(candidate_scores, tie_key)
    base_tp = int(np.asarray(y_true)[base_idx].sum()) if y_true is not None else np.nan
    cand_tp = int(np.asarray(y_true)[cand_idx].sum()) if y_true is not None else np.nan
    tp_gain = cand_tp - base_tp if y_true is not None else np.nan
    one_way_swaps = len(in_only)
    diag = {
        "source": source,
        "arm": arm,
        "capacity": capacity,
        "K": k,
        "overlap": len(base_set & cand_set) / max(k, 1),
        "one_way_swaps": one_way_swaps,
        "churn": one_way_swaps / max(k, 1),
        "tp_gain": tp_gain,
        "tp_gain_per_100_swaps": (100.0 * tp_gain / one_way_swaps) if y_true is not None and one_way_swaps else np.nan,
        "pearson": float(np.corrcoef(base_scores, candidate_scores)[0, 1]),
        "spearman": float(spearmanr(base_scores, candidate_scores).statistic),
    }
    for label, scores, order in (("base", base_scores, base_order), ("candidate", candidate_scores, cand_order)):
        boundary = max(0, k - 1)
        diag[f"{label}_score_rank_k_minus_1"] = float(scores[order[max(0, boundary - 1)]])
        diag[f"{label}_score_rank_k"] = float(scores[order[boundary]])
        diag[f"{label}_score_rank_k_plus_1"] = float(scores[order[min(len(order) - 1, boundary + 1)]])
        diag[f"{label}_boundary_margin"] = float(scores[order[boundary]] - scores[order[min(len(order) - 1, boundary + 1)]])

    swap_rows = []
    for idx in out_only:
        swap_rows.append({
            "source": source, "arm": arm, "direction": "OUT_OF_CANDIDATE", ID_COL: str(ids[idx]),
            "label": int(y_true[idx]) if y_true is not None else np.nan,
            "base_rank": int(base_rank[idx]), "candidate_rank": int(cand_rank[idx]),
            "rank_displacement": int(cand_rank[idx] - base_rank[idx]),
        })
    for idx in in_only:
        swap_rows.append({
            "source": source, "arm": arm, "direction": "INTO_CANDIDATE", ID_COL: str(ids[idx]),
            "label": int(y_true[idx]) if y_true is not None else np.nan,
            "base_rank": int(base_rank[idx]), "candidate_rank": int(cand_rank[idx]),
            "rank_displacement": int(cand_rank[idx] - base_rank[idx]),
        })
    return diag, pd.DataFrame(swap_rows)


def paired_cluster_bootstrap(base_scores, candidate_scores, reps=BOOTSTRAP_REPS):
    # Supplementary conditional uncertainty only: predictions are fixed and models are not refit.
    # Vectorized cluster resampling avoids Python loops over ~149k groups per repetition.
    group_codes, unique_groups = pd.factorize(SIGNATURE_GROUPS, sort=False)
    group_codes = np.asarray(group_codes, dtype=np.int32)
    n_groups = len(unique_groups)
    row_index = np.arange(len(y), dtype=np.int32)
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    deltas = []
    for rep in range(reps):
        sampled_groups = rng.integers(0, n_groups, size=n_groups, dtype=np.int32)
        multiplicity = np.bincount(sampled_groups, minlength=n_groups)
        row_multiplicity = multiplicity[group_codes]
        idx = np.repeat(row_index, row_multiplicity)
        base = normalized_recall_components(y[idx], base_scores[idx], TRAIN_TIE[idx], PRIMARY_CAPACITY)["NR"]
        cand = normalized_recall_components(y[idx], candidate_scores[idx], TRAIN_TIE[idx], PRIMARY_CAPACITY)["NR"]
        deltas.append(cand - base)
    return np.asarray(deltas, dtype=float)


def add_primary_gate_fields(registry, fold_df):
    out = registry.copy()
    base = out.loc[out.arm.eq("A0")].iloc[0]
    base_fold = fold_df.loc[fold_df.arm.eq("A0")].set_index("fold")
    for idx, row in out.iterrows():
        arm = row.arm
        arm_fold = fold_df.loc[fold_df.arm.eq(arm)].set_index("fold")
        deltas = (arm_fold["TP5"] - base_fold["TP5"]).reindex(range(5)).to_numpy(dtype=int)
        out.loc[idx, "delta_TP5_vs_A0"] = row.TP5 - base.TP5
        out.loc[idx, "delta_NR5_vs_A0"] = row.NR5 - base.NR5
        out.loc[idx, "delta_AP_vs_A0"] = row.AveragePrecision - base.AveragePrecision
        out.loc[idx, "delta_Brier_vs_A0"] = row.Brier - base.Brier
        out.loc[idx, "delta_NR3_vs_A0"] = row.NR3 - base.NR3
        out.loc[idx, "delta_NR7_vs_A0"] = row.NR7 - base.NR7
        out.loc[idx, "delta_Protection3_vs_A0"] = row.Protection3 - base.Protection3
        out.loc[idx, "delta_Protection5_vs_A0"] = row.Protection5 - base.Protection5
        out.loc[idx, "delta_Protection7_vs_A0"] = row.Protection7 - base.Protection7
        out.loc[idx, "fold_wins_vs_A0"] = int((deltas > 0).sum())
        out.loc[idx, "worst_fold_delta_TP"] = int(deltas.min())
        out.loc[idx, "fold_delta_TP_vector"] = json.dumps(deltas.tolist())
        if arm == "A0":
            eligible = False
            status = "CONTROL"
        else:
            gates = [
                row.TP5 - base.TP5 >= PRIMARY_GATES["min_delta_tp5"],
                int((deltas > 0).sum()) >= PRIMARY_GATES["min_fold_wins"],
                int(deltas.min()) >= PRIMARY_GATES["worst_fold_delta_tp"],
                row.AveragePrecision - base.AveragePrecision >= PRIMARY_GATES["min_delta_ap"],
                row.NR3 - base.NR3 >= PRIMARY_GATES["min_delta_nr3"],
                row.NR7 - base.NR7 >= PRIMARY_GATES["min_delta_nr7"],
                row.Brier - base.Brier <= PRIMARY_GATES["max_delta_brier"],
                row.Protection5 - base.Protection5 >= PRIMARY_GATES["min_delta_protection5"],
                row.Protection3 - base.Protection3 >= PRIMARY_GATES["min_delta_protection_edge"],
                row.Protection7 - base.Protection7 >= PRIMARY_GATES["min_delta_protection_edge"],
            ]
            eligible = bool(all(gates))
            delta_tp = int(row.TP5 - base.TP5)
            status = "PRIMARY_ELIGIBLE" if eligible else ("MARGINAL_8_19" if 8 <= delta_tp <= 19 else "PRIMARY_REJECTED")
        out.loc[idx, "primary_eligible"] = eligible
        out.loc[idx, "primary_status"] = status
    out["primary_eligible"] = out["primary_eligible"].astype(bool)
    return out


def guardrails_pass(candidate, base, primary=False):
    return bool(
        candidate.AveragePrecision - base.AveragePrecision >= PRIMARY_GATES["min_delta_ap"]
        and candidate.NR3 - base.NR3 >= PRIMARY_GATES["min_delta_nr3"]
        and candidate.NR7 - base.NR7 >= PRIMARY_GATES["min_delta_nr7"]
        and candidate.Brier - base.Brier <= PRIMARY_GATES["max_delta_brier"]
        and candidate.Protection5 - base.Protection5 >= PRIMARY_GATES["min_delta_protection5"]
        and candidate.Protection3 - base.Protection3 >= PRIMARY_GATES["min_delta_protection_edge"]
        and candidate.Protection7 - base.Protection7 >= PRIMARY_GATES["min_delta_protection_edge"]
    )

print("Evaluation and gate helpers defined.")

Evaluation and gate helpers defined.


## Phase 1 — Primary partition

All four preregistered arms run on the same five exact-signature grouped folds. This phase is intentionally complete before any gate is applied, so A3 is evaluated as a preregistered arm rather than invented as a rescue after seeing A2.

In [9]:
PRIMARY_SCORES, PRIMARY_FOLD_METRICS = run_partition(
    "primary", PRIMARY_SPLITS, ["A0", "A1", "A2", "A3"], run_label="run1"
)
PRIMARY_REGISTRY, PRIMARY_FAIRNESS = evaluate_partition_scores(
    "primary", "run1", PRIMARY_SCORES, PRIMARY_FOLD_METRICS
)
PRIMARY_REGISTRY = add_primary_gate_fields(PRIMARY_REGISTRY, PRIMARY_FOLD_METRICS)

# Portfolio and cutoff diagnostics versus A0, OOF and test.
portfolio_rows, swap_frames = [], []
for arm in ("A1", "A2", "A3"):
    diag, swaps = portfolio_diagnostic(
        PRIMARY_SCORES["A0"]["oof"], PRIMARY_SCORES[arm]["oof"], y, TRAIN_TIE,
        train_clean[ID_COL].astype(str).to_numpy(), arm, source="oof", capacity=0.05
    )
    portfolio_rows.append(diag)
    if len(swaps):
        swap_frames.append(swaps)
    test_diag, _ = portfolio_diagnostic(
        PRIMARY_SCORES["A0"]["test"], PRIMARY_SCORES[arm]["test"], None, TEST_TIE,
        test_clean[ID_COL].astype(str).to_numpy(), arm, source="test", capacity=0.05
    )
    portfolio_rows.append(test_diag)
PORTFOLIO_DIAGNOSTICS = pd.DataFrame(portfolio_rows)
PORTFOLIO_DIAGNOSTICS.to_csv(OUTPUT_DIR / "portfolio_churn_primary.csv", index=False)
if swap_frames:
    pd.concat(swap_frames, ignore_index=True).to_csv(OUTPUT_DIR / "portfolio_swap_details_primary.csv", index=False)

# Conditional exact-signature cluster bootstrap is supplementary and does not alter gates.
bootstrap_summary, bootstrap_samples = [], []
for arm in ("A1", "A2", "A3"):
    delta = paired_cluster_bootstrap(PRIMARY_SCORES["A0"]["oof"], PRIMARY_SCORES[arm]["oof"])
    bootstrap_summary.append({
        "arm": arm,
        "reps": len(delta),
        "mean_delta_NR5": float(delta.mean()),
        "median_delta_NR5": float(np.median(delta)),
        "ci_2_5": float(np.quantile(delta, 0.025)),
        "ci_97_5": float(np.quantile(delta, 0.975)),
        "probability_positive": float((delta > 0).mean()),
        "method": "paired exact-signature-cluster bootstrap on fixed OOF predictions; no model refitting",
    })
    bootstrap_samples.extend({"arm": arm, "rep": i, "delta_NR5": float(v)} for i, v in enumerate(delta))
pd.DataFrame(bootstrap_summary).to_csv(OUTPUT_DIR / "paired_cluster_bootstrap.csv", index=False)
pd.DataFrame(bootstrap_samples).to_csv(OUTPUT_DIR / "paired_cluster_bootstrap_samples.csv", index=False)

PRIMARY_REGISTRY.to_csv(OUTPUT_DIR / "primary_candidate_registry.csv", index=False)
PRIMARY_ELIGIBLE_ARMS = PRIMARY_REGISTRY.loc[PRIMARY_REGISTRY.primary_eligible & ~PRIMARY_REGISTRY.arm.eq("A0"), "arm"].tolist()
PRIMARY_MARGINAL_ARMS = PRIMARY_REGISTRY.loc[PRIMARY_REGISTRY.primary_status.eq("MARGINAL_8_19"), "arm"].tolist()
print(PRIMARY_REGISTRY[[
    "arm", "TP5", "NR5", "AveragePrecision", "Brier", "Protection5",
    "delta_TP5_vs_A0", "fold_wins_vs_A0", "worst_fold_delta_TP", "primary_status"
]].to_string(index=False))
print("Primary eligible:", PRIMARY_ELIGIBLE_ARMS)
print("Primary marginal only:", PRIMARY_MARGINAL_ARMS)


=== PRIMARY / run1 / arms=['A0', 'A1', 'A2', 'A3'] ===
[primary/run1] fold 1/5 | build fold-safe representation
  A0:   135.8s | TP5=1568 | NR5=0.976947 | AP=0.826087
  A1:   152.8s | TP5=1577 | NR5=0.982555 | AP=0.828102
  A2:   181.1s | TP5=1572 | NR5=0.979439 | AP=0.827612
  A3:   186.5s | TP5=1573 | NR5=0.980062 | AP=0.827265
[primary/run1] fold 2/5 | build fold-safe representation
  A0:   135.7s | TP5=1564 | NR5=0.974455 | AP=0.826159
  A1:   150.4s | TP5=1574 | NR5=0.980685 | AP=0.829151
  A2:   176.5s | TP5=1577 | NR5=0.982555 | AP=0.830025
  A3:   181.7s | TP5=1578 | NR5=0.983178 | AP=0.830054
[primary/run1] fold 3/5 | build fold-safe representation
  A0:   133.5s | TP5=1533 | NR5=0.958125 | AP=0.816348
  A1:   148.2s | TP5=1546 | NR5=0.966250 | AP=0.819132
  A2:   176.7s | TP5=1547 | NR5=0.966875 | AP=0.818840
  A3:   181.3s | TP5=1549 | NR5=0.968125 | AP=0.819500
[primary/run1] fold 4/5 | build fold-safe representation
  A0:   134.2s | TP5=1544 | NR5=0.971069 | AP=0.819845
 

## Phase 2 — Independent grouped confirmation

Only candidates that clear every primary gate are allowed into the second partition. A0 is rerun as the required apples-to-apples control. If no arm is eligible, this phase records an empty confirmation registry and the notebook proceeds directly toward modeling freeze.

In [10]:
SECONDARY_SCORES = {}
SECONDARY_FOLD_METRICS = pd.DataFrame()
SECONDARY_REGISTRY = pd.DataFrame()
SECONDARY_CONFIRMED_ARMS = []

if PRIMARY_ELIGIBLE_ARMS:
    SECONDARY_SCORES, SECONDARY_FOLD_METRICS = run_partition(
        "secondary", SECONDARY_SPLITS, ["A0"] + PRIMARY_ELIGIBLE_ARMS, run_label="run1"
    )
    secondary_base_registry, _ = evaluate_partition_scores(
        "secondary", "run1", SECONDARY_SCORES, SECONDARY_FOLD_METRICS
    )
    sec_base = secondary_base_registry.loc[secondary_base_registry.arm.eq("A0")].iloc[0]
    pri_lookup = PRIMARY_REGISTRY.set_index("arm")
    sec_fold_base = SECONDARY_FOLD_METRICS.loc[SECONDARY_FOLD_METRICS.arm.eq("A0")].set_index("fold")
    rows = []
    for _, row in secondary_base_registry.iterrows():
        arm = row.arm
        arm_fold = SECONDARY_FOLD_METRICS.loc[SECONDARY_FOLD_METRICS.arm.eq(arm)].set_index("fold")
        deltas = (arm_fold["TP5"] - sec_fold_base["TP5"]).reindex(range(5)).to_numpy(dtype=int)
        if arm == "A0":
            confirmed = False
            total_wins = 0
            mean_gain = 0.0
            status = "CONTROL"
        else:
            primary_row = pri_lookup.loc[arm]
            primary_wins = int(primary_row.fold_wins_vs_A0)
            secondary_wins = int((deltas > 0).sum())
            total_wins = primary_wins + secondary_wins
            primary_gain = float(primary_row.delta_TP5_vs_A0)
            secondary_gain = float(row.TP5 - sec_base.TP5)
            mean_gain = (primary_gain + secondary_gain) / 2.0
            confirmed = bool(
                secondary_gain >= SECONDARY_GATES["min_delta_tp5"]
                and secondary_wins >= SECONDARY_GATES["min_fold_wins"]
                and int(deltas.min()) >= SECONDARY_GATES["worst_fold_delta_tp"]
                and total_wins >= SECONDARY_GATES["min_total_fold_wins"]
                and mean_gain >= SECONDARY_GATES["min_mean_tp_gain"]
                and guardrails_pass(row, sec_base)
            )
            status = "SECONDARY_CONFIRMED" if confirmed else ("SECONDARY_MIXED" if secondary_gain > 0 else "SECONDARY_REJECTED")
        record = row.to_dict()
        record.update({
            "delta_TP5_vs_A0": float(row.TP5 - sec_base.TP5),
            "fold_wins_vs_A0": int((deltas > 0).sum()) if arm != "A0" else 0,
            "worst_fold_delta_TP": int(deltas.min()) if arm != "A0" else 0,
            "fold_delta_TP_vector": json.dumps(deltas.tolist()),
            "total_fold_wins_primary_secondary": total_wins,
            "mean_TP_gain_primary_secondary": mean_gain,
            "secondary_confirmed": confirmed,
            "secondary_status": status,
        })
        rows.append(record)
    SECONDARY_REGISTRY = pd.DataFrame(rows)
    SECONDARY_REGISTRY["secondary_confirmed"] = SECONDARY_REGISTRY["secondary_confirmed"].astype(bool)
    SECONDARY_REGISTRY.to_csv(OUTPUT_DIR / "secondary_candidate_registry.csv", index=False)
    SECONDARY_CONFIRMED_ARMS = SECONDARY_REGISTRY.loc[
        SECONDARY_REGISTRY.secondary_confirmed & ~SECONDARY_REGISTRY.arm.eq("A0"), "arm"
    ].tolist()
else:
    pd.DataFrame(columns=["arm", "secondary_confirmed", "secondary_status"]).to_csv(
        OUTPUT_DIR / "secondary_candidate_registry.csv", index=False
    )

print("Secondary confirmed:", SECONDARY_CONFIRMED_ARMS)
if len(SECONDARY_REGISTRY):
    print(SECONDARY_REGISTRY[[
        "arm", "TP5", "NR5", "delta_TP5_vs_A0", "fold_wins_vs_A0",
        "worst_fold_delta_TP", "total_fold_wins_primary_secondary", "secondary_status"
    ]].to_string(index=False))


=== SECONDARY / run1 / arms=['A0', 'A1', 'A2', 'A3'] ===
[secondary/run1] fold 1/5 | build fold-safe representation
  A0:   132.9s | TP5=1568 | NR5=0.975731 | AP=0.828700
  A1:   147.6s | TP5=1576 | NR5=0.980709 | AP=0.831236
  A2:   177.7s | TP5=1573 | NR5=0.978843 | AP=0.829734
  A3:   181.9s | TP5=1576 | NR5=0.980709 | AP=0.829366
[secondary/run1] fold 2/5 | build fold-safe representation
  A0:   133.5s | TP5=1546 | NR5=0.965646 | AP=0.821590
  A1:   147.9s | TP5=1557 | NR5=0.972517 | AP=0.825213
  A2:   176.1s | TP5=1556 | NR5=0.971893 | AP=0.825112
  A3:   180.3s | TP5=1559 | NR5=0.973766 | AP=0.825263
[secondary/run1] fold 3/5 | build fold-safe representation
  A0:   132.4s | TP5=1559 | NR5=0.972551 | AP=0.824613
  A1:   147.9s | TP5=1564 | NR5=0.975671 | AP=0.827757
  A2:   176.3s | TP5=1563 | NR5=0.975047 | AP=0.826448
  A3:   181.1s | TP5=1564 | NR5=0.975671 | AP=0.826116
[secondary/run1] fold 4/5 | build fold-safe representation
  A0:   132.1s | TP5=1555 | NR5=0.975533 | AP=

In [11]:
# Frozen complexity selection: simplest confirmed arm wins unless added complexity buys >=8 primary TP and non-negative secondary TP.
def choose_confirmed_challenger(confirmed_arms):
    confirmed_arms = sorted(confirmed_arms, key=lambda a: ARM_SPECS[a]["level"])
    if not confirmed_arms:
        return None, []
    pri = PRIMARY_REGISTRY.set_index("arm")
    sec = SECONDARY_REGISTRY.set_index("arm") if len(SECONDARY_REGISTRY) else None
    selected = confirmed_arms[0]
    decisions = [{"candidate": selected, "against": None, "selected": True, "reason": "simplest secondary-confirmed candidate"}]
    for arm in confirmed_arms[1:]:
        primary_increment = float(pri.loc[arm, "TP5"] - pri.loc[selected, "TP5"])
        secondary_increment = float(sec.loc[arm, "TP5"] - sec.loc[selected, "TP5"])
        displace = bool(
            primary_increment >= COMPLEXITY_GATES["min_incremental_primary_tp"]
            and secondary_increment >= COMPLEXITY_GATES["min_incremental_secondary_tp"]
        )
        decisions.append({
            "candidate": arm,
            "against": selected,
            "primary_incremental_TP": primary_increment,
            "secondary_incremental_TP": secondary_increment,
            "selected": displace,
            "reason": "incremental complexity gate passed" if displace else "extra complexity did not buy enough transferable TP",
        })
        if displace:
            selected = arm
    return selected, decisions

FINAL_CHALLENGER, COMPLEXITY_DECISIONS = choose_confirmed_challenger(SECONDARY_CONFIRMED_ARMS)
pd.DataFrame(COMPLEXITY_DECISIONS).to_csv(OUTPUT_DIR / "complexity_selection.csv", index=False)
print("Final challenger before stochastic rerun:", FINAL_CHALLENGER)
if COMPLEXITY_DECISIONS:
    print(pd.DataFrame(COMPLEXITY_DECISIONS).to_string(index=False))

Final challenger before stochastic rerun: A1
candidate against  selected                                              reason  primary_incremental_TP  secondary_incremental_TP
       A1    None      True              simplest secondary-confirmed candidate                     NaN                       NaN
       A2      A1     False extra complexity did not buy enough transferable TP                    -7.0                      -5.0
       A3      A1     False extra complexity did not buy enough transferable TP                    -9.0                       1.0


## Phase 3 — Same-seed GPU stochastic rerun

Only A0 and the final confirmed challenger are rerun. Seeds, feature definitions, folds, software, and GPU topology are unchanged. The rerun is a falsification check for CatBoost GPU stochasticity, not a seed search and not a second chance to select another arm.

In [12]:
RERUN_SCORES = {}
RERUN_FOLD_METRICS = pd.DataFrame()
RERUN_REGISTRY = pd.DataFrame()
RERUN_PASS = False
RERUN_STABILITY = pd.DataFrame()

if FINAL_CHALLENGER is not None:
    RERUN_SCORES, RERUN_FOLD_METRICS = run_partition(
        "primary", PRIMARY_SPLITS, ["A0", FINAL_CHALLENGER], run_label="rerun"
    )
    rerun_base_registry, _ = evaluate_partition_scores(
        "primary", "rerun", RERUN_SCORES, RERUN_FOLD_METRICS
    )
    rr_base = rerun_base_registry.loc[rerun_base_registry.arm.eq("A0")].iloc[0]
    rr_cand = rerun_base_registry.loc[rerun_base_registry.arm.eq(FINAL_CHALLENGER)].iloc[0]
    rr_base_fold = RERUN_FOLD_METRICS.loc[RERUN_FOLD_METRICS.arm.eq("A0")].set_index("fold")
    rr_cand_fold = RERUN_FOLD_METRICS.loc[RERUN_FOLD_METRICS.arm.eq(FINAL_CHALLENGER)].set_index("fold")
    rr_delta = (rr_cand_fold["TP5"] - rr_base_fold["TP5"]).reindex(range(5)).to_numpy(dtype=int)

    # Repeat stability is measured separately for control and challenger on OOF top-5 and test top-2002.
    a0_oof_diag, _ = portfolio_diagnostic(
        PRIMARY_SCORES["A0"]["oof"], RERUN_SCORES["A0"]["oof"], y, TRAIN_TIE,
        train_clean[ID_COL].astype(str).to_numpy(), "A0_repeat", source="oof_repeat", capacity=0.05
    )
    cand_oof_diag, _ = portfolio_diagnostic(
        PRIMARY_SCORES[FINAL_CHALLENGER]["oof"], RERUN_SCORES[FINAL_CHALLENGER]["oof"], y, TRAIN_TIE,
        train_clean[ID_COL].astype(str).to_numpy(), f"{FINAL_CHALLENGER}_repeat", source="oof_repeat", capacity=0.05
    )
    a0_test_diag, _ = portfolio_diagnostic(
        PRIMARY_SCORES["A0"]["test"], RERUN_SCORES["A0"]["test"], None, TEST_TIE,
        test_clean[ID_COL].astype(str).to_numpy(), "A0_repeat", source="test_repeat", capacity=0.05
    )
    cand_test_diag, _ = portfolio_diagnostic(
        PRIMARY_SCORES[FINAL_CHALLENGER]["test"], RERUN_SCORES[FINAL_CHALLENGER]["test"], None, TEST_TIE,
        test_clean[ID_COL].astype(str).to_numpy(), f"{FINAL_CHALLENGER}_repeat", source="test_repeat", capacity=0.05
    )
    RERUN_STABILITY = pd.DataFrame([a0_oof_diag, cand_oof_diag, a0_test_diag, cand_test_diag])
    RERUN_STABILITY.to_csv(OUTPUT_DIR / "rerun_overlap.csv", index=False)

    a0_oof_overlap = float(a0_oof_diag["overlap"])
    cand_oof_overlap = float(cand_oof_diag["overlap"])
    a0_test_overlap = float(a0_test_diag["overlap"])
    cand_test_overlap = float(cand_test_diag["overlap"])
    min_cand_oof_overlap = max(RERUN_GATES["min_repeat_overlap"], a0_oof_overlap - RERUN_GATES["max_relative_overlap_loss"])
    min_cand_test_overlap = max(RERUN_GATES["min_repeat_overlap"], a0_test_overlap - RERUN_GATES["max_relative_overlap_loss"])

    RERUN_PASS = bool(
        (rr_cand.TP5 - rr_base.TP5) >= RERUN_GATES["min_delta_tp5"]
        and int((rr_delta > 0).sum()) >= RERUN_GATES["min_fold_wins"]
        and int(rr_delta.min()) >= RERUN_GATES["worst_fold_delta_tp"]
        and guardrails_pass(rr_cand, rr_base)
        and a0_oof_overlap >= RERUN_GATES["min_repeat_overlap"]
        and a0_test_overlap >= RERUN_GATES["min_repeat_overlap"]
        and cand_oof_overlap >= min_cand_oof_overlap
        and cand_test_overlap >= min_cand_test_overlap
    )
    RERUN_REGISTRY = rerun_base_registry.copy()
    RERUN_REGISTRY["rerun_delta_TP5_vs_A0"] = RERUN_REGISTRY.TP5 - rr_base.TP5
    RERUN_REGISTRY["rerun_pass_final_challenger"] = False
    RERUN_REGISTRY.loc[RERUN_REGISTRY.arm.eq(FINAL_CHALLENGER), "rerun_pass_final_challenger"] = RERUN_PASS
    RERUN_REGISTRY.to_csv(OUTPUT_DIR / "rerun_candidate_registry.csv", index=False)
    print(RERUN_REGISTRY[["arm", "TP5", "NR5", "AveragePrecision", "Brier", "Protection5", "rerun_delta_TP5_vs_A0"]].to_string(index=False))
    print("Rerun fold TP delta:", rr_delta.tolist())
    print("Rerun stability:")
    print(RERUN_STABILITY[["source", "arm", "overlap", "churn", "spearman"]].to_string(index=False))
    print("RERUN PASS:", RERUN_PASS)
else:
    pd.DataFrame(columns=["arm", "rerun_pass_final_challenger"]).to_csv(OUTPUT_DIR / "rerun_candidate_registry.csv", index=False)
    pd.DataFrame(columns=["source", "arm", "overlap"]).to_csv(OUTPUT_DIR / "rerun_overlap.csv", index=False)
    print("No secondary-confirmed challenger: stochastic rerun skipped by contract.")


=== PRIMARY / rerun / arms=['A0', 'A1'] ===
[primary/rerun] fold 1/5 | build fold-safe representation
  A0:   134.1s | TP5=1561 | NR5=0.972586 | AP=0.825611
  A1:   149.2s | TP5=1574 | NR5=0.980685 | AP=0.828511
[primary/rerun] fold 2/5 | build fold-safe representation
  A0:   133.4s | TP5=1566 | NR5=0.975701 | AP=0.826427
  A1:   148.5s | TP5=1572 | NR5=0.979439 | AP=0.829061
[primary/rerun] fold 3/5 | build fold-safe representation
  A0:   133.5s | TP5=1536 | NR5=0.960000 | AP=0.815827
  A1:   149.3s | TP5=1545 | NR5=0.965625 | AP=0.819077
[primary/rerun] fold 4/5 | build fold-safe representation
  A0:   134.1s | TP5=1546 | NR5=0.972327 | AP=0.820072
  A1:   148.8s | TP5=1546 | NR5=0.972327 | AP=0.823107
[primary/rerun] fold 5/5 | build fold-safe representation
  A0:   133.6s | TP5=1557 | NR5=0.970093 | AP=0.825418
  A1:   152.6s | TP5=1563 | NR5=0.973832 | AP=0.828802
arm  TP5      NR5  AveragePrecision    Brier  Protection5  rerun_delta_TP5_vs_A0
 A0 7764 0.969530          0.82267

## Final frozen decision and conditional submission

The public leaderboard is not consulted here. A submission is produced only when a candidate survives primary, independent confirmation, complexity selection, stochastic rerun, all guardrails, and reaches the predeclared historical V3 near-parity region on the primary OOF context. The submission uses the **first primary run's five-fold mean raw CatBoost probabilities**; the stochastic rerun is confirmation only and is never cherry-picked.

In [13]:
FINAL_PROMOTED = False
INCUMBENT_CONTEXT = "NO_CHALLENGER"
FINAL_REASON = "No V10 candidate passed the full offline contract."

if FINAL_CHALLENGER is not None:
    challenger_primary = PRIMARY_REGISTRY.loc[PRIMARY_REGISTRY.arm.eq(FINAL_CHALLENGER)].iloc[0]
    tp5 = int(challenger_primary.TP5)
    if tp5 >= V3_ORIGINAL_TP5:
        INCUMBENT_CONTEXT = "V3_PARITY_OR_WIN_REGION"
    elif tp5 >= V3_NEAR_PARITY_TP5:
        INCUMBENT_CONTEXT = "V3_NEAR_PARITY_REGION"
    else:
        INCUMBENT_CONTEXT = "BELOW_V3_CHAMPION_REGION"

    FINAL_PROMOTED = bool(RERUN_PASS and tp5 >= V3_NEAR_PARITY_TP5)
    if FINAL_PROMOTED:
        FINAL_REASON = (
            f"{FINAL_CHALLENGER} passed primary, secondary, complexity, stochastic stability, guardrail, "
            f"and V3-context gates ({tp5} primary TP@5%)."
        )
    elif not RERUN_PASS:
        FINAL_REASON = f"{FINAL_CHALLENGER} did not survive the frozen GPU stochastic rerun gate."
    else:
        FINAL_REASON = (
            f"{FINAL_CHALLENGER} improved A0 but remained below the predeclared V3 near-parity region "
            f"({tp5} < {V3_NEAR_PARITY_TP5} TP@5%)."
        )

if FINAL_PROMOTED:
    prediction = np.clip(PRIMARY_SCORES[FINAL_CHALLENGER]["test"], 0.0, 1.0)
    submission = pd.DataFrame({
        ID_COL: sample_submission[ID_COL].astype(str),
        "fraud_probability": prediction,
    })
    if list(submission.columns) != [ID_COL, "fraud_probability"]:
        raise RuntimeError("Submission schema invalid.")
    if not np.isfinite(submission["fraud_probability"]).all() or not submission["fraud_probability"].between(0, 1).all():
        raise RuntimeError("Submission probabilities invalid.")
    if submission[ID_COL].tolist() != sample_submission[ID_COL].astype(str).tolist():
        raise RuntimeError("Submission ID order differs from sample_submission.")
    submission.to_csv(OUTPUT_DIR / "submission_v10.csv", index=False)
    FINAL_CONCLUSION = f"V10 offline promoted: {FINAL_CHALLENGER}. One leaderboard submission is permissible for external confirmation; no leaderboard-driven retuning."
else:
    FINAL_CONCLUSION = "Modeling freeze. V3 remains champion."
    (OUTPUT_DIR / "NO_SUBMISSION.txt").write_text(FINAL_CONCLUSION + "\n" + FINAL_REASON + "\n")

SELECTED_CONFIG = {
    "final_challenger": FINAL_CHALLENGER,
    "final_promoted": FINAL_PROMOTED,
    "incumbent_context": INCUMBENT_CONTEXT,
    "final_reason": FINAL_REASON,
    "conclusion": FINAL_CONCLUSION,
    "submission_source": "primary run1 five-fold probability mean" if FINAL_PROMOTED else None,
    "frozen_contract_sha256": FROZEN_CONTRACT_HASH,
    "resolved_experiment_sha256": RESOLVED_EXPERIMENT_HASH,
}
(OUTPUT_DIR / "selected_config.json").write_text(json.dumps(SELECTED_CONFIG, indent=2))
(OUTPUT_DIR / "research_conclusion.json").write_text(json.dumps({
    **SELECTED_CONFIG,
    "primary_eligible_arms": PRIMARY_ELIGIBLE_ARMS,
    "primary_marginal_arms": PRIMARY_MARGINAL_ARMS,
    "secondary_confirmed_arms": SECONDARY_CONFIRMED_ARMS,
    "complexity_decisions": COMPLEXITY_DECISIONS,
    "rerun_pass": RERUN_PASS,
}, indent=2, default=str))
print("\nFINAL DECISION")
print("==============")
print(FINAL_CONCLUSION)
print(FINAL_REASON)
print("Incumbent context:", INCUMBENT_CONTEXT)


FINAL DECISION
Modeling freeze. V3 remains champion.
A1 improved A0 but remained below the predeclared V3 near-parity region (7814 < 7820 TP@5%).
Incumbent context: BELOW_V3_CHAMPION_REGION


In [14]:

# Consolidated audit/report artifacts required by the frozen notebook delivery contract.
# Primary 3/5/7 operating table.
capacity_rows = []
for _, row in PRIMARY_REGISTRY.iterrows():
    for cap in (3, 5, 7):
        capacity_rows.append({
            "partition": "primary", "run": "run1", "arm": row.arm, "capacity": cap / 100,
            "K": int(row[f"K{cap}"]), "TP": int(row[f"TP{cap}"]), "NR": float(row[f"NR{cap}"]),
            "precision": float(row[f"Precision{cap}"]), "legitimate_audits": int(row[f"LegitimateAudits{cap}"]),
            "ProtectionScore": float(row[f"Protection{cap}"]), "Gap_gender": float(row[f"GapGender{cap}"]),
            "Gap_age": float(row[f"GapAge{cap}"]),
        })
CAPACITY_TABLE = pd.DataFrame(capacity_rows)
CAPACITY_TABLE.to_csv(OUTPUT_DIR / "capacity_3_5_7.csv", index=False)
PRIMARY_FAIRNESS.to_csv(OUTPUT_DIR / "fairness_3_5_7.csv", index=False)

CALIBRATION_DIAGNOSTICS = PRIMARY_REGISTRY[["arm", "AveragePrecision", "Brier"]].copy()
CALIBRATION_DIAGNOSTICS["mean_predicted_probability"] = [
    float(np.mean(PRIMARY_SCORES[a]["oof"])) for a in CALIBRATION_DIAGNOSTICS.arm
]
CALIBRATION_DIAGNOSTICS["observed_prevalence"] = float(np.mean(y))
CALIBRATION_DIAGNOSTICS["calibration_in_the_large_error"] = (
    CALIBRATION_DIAGNOSTICS["mean_predicted_probability"] - CALIBRATION_DIAGNOSTICS["observed_prevalence"]
)
CALIBRATION_DIAGNOSTICS.to_csv(OUTPUT_DIR / "calibration_diagnostics.csv", index=False)

PORTFOLIO_DIAGNOSTICS.loc[PORTFOLIO_DIAGNOSTICS.source.eq("oof")].to_csv(
    OUTPUT_DIR / "boundary_diagnostics.csv", index=False
)
PORTFOLIO_DIAGNOSTICS.loc[PORTFOLIO_DIAGNOSTICS.source.eq("test")].to_csv(
    OUTPUT_DIR / "test_churn_primary.csv", index=False
)
if len(RERUN_STABILITY):
    RERUN_STABILITY.loc[RERUN_STABILITY.source.eq("test_repeat")].to_csv(
        OUTPUT_DIR / "rerun_test_churn.csv", index=False
    )
else:
    pd.DataFrame(columns=["source", "arm", "overlap", "churn", "spearman"]).to_csv(
        OUTPUT_DIR / "rerun_test_churn.csv", index=False
    )

# Fold TP deltas against the appropriate A0 control.
fold_delta_frames = []
for partition, run_label, fold_df in [
    ("primary", "run1", PRIMARY_FOLD_METRICS),
    ("secondary", "run1", SECONDARY_FOLD_METRICS),
    ("primary", "rerun", RERUN_FOLD_METRICS),
]:
    if fold_df is None or len(fold_df) == 0:
        continue
    base = fold_df.loc[fold_df.arm.eq("A0"), ["fold", "TP5"]].rename(columns={"TP5": "A0_TP5"})
    tmp = fold_df.merge(base, on="fold", how="left", validate="many_to_one")
    tmp["delta_TP5_vs_A0"] = tmp["TP5"] - tmp["A0_TP5"]
    fold_delta_frames.append(tmp)
FOLD_TP_DELTAS = pd.concat(fold_delta_frames, ignore_index=True) if fold_delta_frames else pd.DataFrame()
FOLD_TP_DELTAS.to_csv(OUTPUT_DIR / "fold_tp_deltas.csv", index=False)

# Consolidated support/fallback diagnostics, if A2/A3 were materialized in each stage.
def _concat_existing(pattern, output_name):
    paths = sorted(OUTPUT_DIR.glob(pattern))
    frames = [pd.read_csv(path) for path in paths if path.stat().st_size > 0]
    if frames:
        pd.concat(frames, ignore_index=True).to_csv(OUTPUT_DIR / output_name, index=False)
    else:
        pd.DataFrame().to_csv(OUTPUT_DIR / output_name, index=False)

_concat_existing("peer_support_profile_*_*.csv", "peer_support_profile.csv")
_concat_existing("fallback_usage_*_*.csv", "fallback_usage.csv")

# One audit table spanning every promotion stage.
promotion_rows = []
for _, row in PRIMARY_REGISTRY.iterrows():
    promotion_rows.append({
        "arm": row.arm,
        "primary_status": row.primary_status,
        "primary_eligible": bool(row.primary_eligible),
        "primary_delta_TP5": float(row.delta_TP5_vs_A0),
        "primary_fold_wins": int(row.fold_wins_vs_A0),
        "primary_worst_fold_delta_TP": int(row.worst_fold_delta_TP),
        "secondary_status": None,
        "secondary_confirmed": False,
        "rerun_pass": False,
        "final_selected_challenger": bool(row.arm == FINAL_CHALLENGER),
        "final_promoted": bool(FINAL_PROMOTED and row.arm == FINAL_CHALLENGER),
    })
if len(SECONDARY_REGISTRY):
    sec = SECONDARY_REGISTRY.set_index("arm")
    for rec in promotion_rows:
        if rec["arm"] in sec.index:
            s = sec.loc[rec["arm"]]
            rec["secondary_status"] = s.secondary_status
            rec["secondary_confirmed"] = bool(s.secondary_confirmed)
for rec in promotion_rows:
    if rec["arm"] == FINAL_CHALLENGER:
        rec["rerun_pass"] = bool(RERUN_PASS)
PROMOTION_GATE_RESULTS = pd.DataFrame(promotion_rows)
PROMOTION_GATE_RESULTS.to_csv(OUTPUT_DIR / "promotion_gate_results.csv", index=False)
print("Consolidated report/audit artifacts written.")


Consolidated report/audit artifacts written.


In [15]:
# Final artifact manifest and portable ZIP. This cell runs whether V10 wins or freezes.
def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

manifest_rows = []
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file() and path.name != "artifact_manifest.json":
        manifest_rows.append({
            "path": str(path.relative_to(OUTPUT_DIR)),
            "bytes": path.stat().st_size,
            "sha256": file_sha256(path),
        })
ARTIFACT_MANIFEST = {
    "frozen_contract_sha256": FROZEN_CONTRACT_HASH,
    "resolved_experiment_sha256": RESOLVED_EXPERIMENT_HASH,
    "final_promoted": FINAL_PROMOTED,
    "final_challenger": FINAL_CHALLENGER,
    "files": manifest_rows,
}
(OUTPUT_DIR / "artifact_manifest.json").write_text(json.dumps(ARTIFACT_MANIFEST, indent=2))
ZIP_PATH = Path("/kaggle/working/v10_artifacts.zip")
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
shutil.make_archive(str(ZIP_PATH.with_suffix("")), "zip", root_dir=OUTPUT_DIR)
print("Artifacts:", len(manifest_rows) + 1, "files")
print("ZIP:", ZIP_PATH)
print("Final conclusion:", FINAL_CONCLUSION)

Artifacts: 112 files
ZIP: /kaggle/working/v10_artifacts.zip
Final conclusion: Modeling freeze. V3 remains champion.


## Interpretation boundary

- A V10 local gain is accepted only after the frozen multi-stage evidence gate. Small `+1` to `+7` TP changes are treated as cutoff noise; `+8` to `+19` TP is marginal and non-promotable.
- A strong primary result that fails the independent grouped partition or same-seed GPU rerun is a falsified transfer/stability hypothesis, not a reason to tune another V10 variant.
- Brier and Protection Score are guardrails, not alternate ranking-selection objectives.
- The ~50/50 competition prevalence is not an operational NHPA fraud base rate, so raw CatBoost probabilities must not be interpreted as production-calibrated fraud prevalence without prospective recalibration.
- No V11 is created by this notebook. If V10 does not pass, the research conclusion is **Modeling freeze. V3 remains champion.**